# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
print("Available Record Sets in the dataset:\n")
record_sets = []
for record_set in dataset.record_sets:
    print(f"- name: {record_set.name}")
    print(f"  @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()
    record_sets.append(record_set.id)

# Optionally, examine one record from each set
print("Sample record from each record set:")
for rec_set_id in record_sets:
    try:
        rec_list = list(dataset.records(record_set=rec_set_id))
        print(f"Sample from {rec_set_id}:", rec_list[0] if rec_list else 'No records')
    except Exception as e:
        print(f"Could not sample from {rec_set_id}: {e}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose one or more record sets (from their @id)
# For this dataset, let's list all record set @id (as found previously):
print(f"Record Sets found: {record_sets}")

# You may update this list based on overview
record_sets_to_load = record_sets  # Load all for now

dataframes = {}
for record_set in record_sets_to_load:
    records = list(dataset.records(record_set=record_set))
    df = pd.DataFrame(records)
    dataframes[record_set] = df
    print(f"Loaded {len(df)} records for record_set: {record_set}")

# Preview columns of the first record set with data
for rec_set_id in dataframes:
    df = dataframes[rec_set_id]
    if len(df) > 0:
        print(f"\nColumns for record_set {rec_set_id}: {df.columns.tolist()}")
        display(df.head())
        # Assign for EDA/next analysis
        primary_record_set_id = rec_set_id
        break

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Get the DataFrame for EDA
df = dataframes[primary_record_set_id]
print(f"Rows: {len(df)} Columns: {list(df.columns)}\n")

# Try to select a numeric field for analysis
numeric_candidates = [col for col in df.columns if pd.api.types.is_numeric_dtype(df[col])]
print("Numeric fields detected:", numeric_candidates)

if numeric_candidates:
    numeric_field = numeric_candidates[0]  # Use the first detected numeric field
    threshold = df[numeric_field].mean() if df[numeric_field].mean() > 0 else 10
    filtered_df = df[df[numeric_field] > threshold].copy()
    print(f"Filtered records with {numeric_field} > {threshold}:")
    display(filtered_df.head())

    # Normalize
    filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
    print(f"Normalized {numeric_field} for filtered records:")
    display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

    # Try grouping by another field (categorical)
    categorical_candidates = [col for col in df.columns if pd.api.types.is_string_dtype(df[col]) and df[col].nunique() < 10]
    if categorical_candidates:
        group_field = categorical_candidates[0]
        grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
        print(f"Grouped average {numeric_field} by {group_field}:")
        display(grouped_df)
else:
    print("No numeric fields detected for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution & normalized values if available
if numeric_candidates:
    plt.figure(figsize=(10,5))
    sns.histplot(df[numeric_field], kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field} in {primary_record_set_id}")
    plt.xlabel(numeric_field)
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df.columns:
        plt.figure(figsize=(10,5))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"], kde=True, bins=30)
        plt.title(f"Distribution of {numeric_field} (Normalized) in filtered records")
        plt.xlabel(f"{numeric_field}_normalized")
        plt.show()

    # If grouping is done, show barplot
    if 'grouped_df' in locals():
        plt.figure(figsize=(8,5))
        sns.barplot(data=grouped_df, x=group_field, y=numeric_field, palette='muted')
        plt.title(f"Mean {numeric_field} grouped by {group_field}")
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

In this notebook, we loaded the FAIR² Croissant dataset on human mast cell extracellular vesicle mass spectrometry, explored available record sets and fields using their `@id`s, loaded and previewed data, performed basic EDA filtering and normalization using detected numeric fields, and visualized key data distributions.

To further analyze the dataset, consider exploring relationships among more fields or integrating domain-specific scientific questions.